# Knapsack 0/1 problem.
---
Description:

- Optimization (max)
- Single-objective
- Constraints (yes)
---

The 0/1 Knapsack Problem is a classic combinatorial optimization challenge. In mathematical temrs is structured
as a constrained maximization problem:

- maximize: $\sum _{i=1}^{n}v_{i}x_{i}$

- subject to: $\sum _{i=1}^{n}w_{i}x_{i}\le W_{max}$

- where: $x_{i} \in \{0, 1\} \quad \forall i\in \{1,\dots ,n\}$

### First we import Python libraries and set up the directory of our code.

In [1]:
import os, sys
import numpy as np
from math import fsum

PROJECT_DIR = os.path.abspath('..')

if PROJECT_DIR not in sys.path:
    sys.path.append(PROJECT_DIR)

### Here we import all our custom GA code.

In [2]:
# Import main classes.
from pygenalgo.genome.gene import Gene
from pygenalgo.genome.chromosome import Chromosome
from pygenalgo.utils.utilities import cost_function
from pygenalgo.engines.standard_ga import StandardGA, RunConfig

# Import Selection Operator(s).
from pygenalgo.operators.selection.linear_rank_selector import LinearRankSelector

# Import Crossover Operator(s).
from pygenalgo.operators.crossover.uniform_crossover import UniformCrossover

# Import Mutation Operator(s).
from pygenalgo.operators.mutation.flip_mutator import FlipMutator

### Define the Knapsack 0/1 fitness function.

In [3]:
# Knapsack function.
@cost_function
def fun_knapsack(individual: Chromosome, params: dict):
    # Extract the parameters.
    item_weights = params["weights"]
    item_values = params["values"]
    max_weight = params["max_weight"]

    # Extract the gene values.
    x_values = individual.values()
    
    # Calculate total weight and raw value in a single loop.
    total_weight, f_value = 0, 0
    for x, w, v in zip(x_values, item_weights, item_values):
        # Only add if the item is selected.
        if x == 1:
            total_weight += w
            f_value += v
    
    # Compute the constraint.
    c1: float = max(0.0, total_weight - max_weight)**2

    # Set a penalty coefficient.
    rho: float = 100.0
    
    # Return the f(x) - penalty.
    return f_value - (rho * c1)
# _end_def_

Here we set the GA parameters, such as number of genes, number of chromosomes, etc.

In [7]:
# Random number generator.
rng = np.random.default_rng()

# Random function: It is used only for compatibility.
boundary_x = lambda: rng.integers(2)

# Setup an example.
params = {
    "weights": [
        45, 12,  6, 22, 20, 19, 13, 11, 48, 39,
        10, 42, 32,  7,  6, 10, 18, 19, 37, 43,
         6, 40, 17, 50, 46, 49, 39, 31, 19, 33,
        42, 22,  5, 15, 49, 32, 26, 22, 14, 18,
        26, 11, 10, 29, 11, 27, 27, 43, 21,  7
    ],
    "values": [
        228,  25,  15,  93, 120, 112,  21,  32,  94,  79,
         42, 165, 128,  26,  29,  24,  57,  54, 186, 177,
         34, 155,  46, 201, 180, 151, 154, 156,  92, 129,
        129, 131,  31,  91, 194,  99, 126,  40,  47,  59,
        134,  23,  54, 148,  47,  59,  59, 257,  80,  29
    ],
    "max_weight": 400
}

# Define the number of genes (items).
M = len(params["weights"])

# Define the number of chromosomes.
N = 100

# Draw random samples for the initial points.
x_init = rng.integers(low=0, high=2, size=(N, M))


# Initial population.
population = [Chromosome([Gene(x_init[i, j], boundary_x)
                          for j in range(M)], None, True)
              for i in range(N)]

# Create the StandardGA object that will carry on the optimization.
test_GA = StandardGA(initial_pop=population,
                     fit_func=lambda x: fun_knapsack(x, params),
                     select_op=LinearRankSelector(eta=1.5),
                     mutate_op=FlipMutator(),
                     crossx_op=UniformCrossover())

### Optimization process.

Here we call the GA object. We set a number of parameters, such as the maximum iterations (i.e. epochs), tolerance for the fitness convergence, etc.

In [8]:
test_GA(config=RunConfig(epochs=250, elitism=True, verbose=False))

09/14/2026 14:23:56 INFO: Initial Avg. Fitness = -5671146.7700
09/14/2026 14:23:58 INFO: Final: Avg. Fitness = 1588.3400


Elapsed time: 2.197 seconds.


In [9]:
# Extract the optimal solution from the GA.
optimal_solution = test_GA.best_chromosome()

# Extract the parameters.
item_values = params["values"]
item_weights = params["weights"]
max_weight = params["max_weight"]

# Extract the gene values.
x_values = optimal_solution.values()

# Calculate total weight and raw value in a single loop
total_weight, f_value = 0, 0
for x, w, v in zip(x_values, item_weights, item_values):
    # Only add if the item is selected.
    if x == 1:
        total_weight += w
        f_value += v

# Display the (final) optimal value.
print(f"Optimum Found: {f_value}, Total weight: {total_weight}\n")

# The optimal solution for this parameter setting is: (2098, 399).

Optimum Found: 2098, Total weight: 399



### End of file